# Grafos dos paths - Set12 07.png, sal-e-pimenta medium

Este notebook carrega o pickle ja pronto do experimento sal-e-pimenta `medium`, seleciona tres centros impulsivos em regioes lisa, mista e densa, e visualiza os grafos k-NN/paths usados pelo algoritmo hibrido com `f=1`, `t=3`, `nn=7` e multiplicador `h=0.001`.

In [ ]:
from pathlib import Path
import sys
import pickle

PROJECT_ROOT = Path('/workspace')
sys.path.append(str(PROJECT_ROOT / 'src' / 'salt_experiments'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import networkx as nx
import sklearn.neighbors as sknn
from sklearn.decomposition import PCA
from scipy.ndimage import generic_filter

from functions.nlm_functions import mirror_cpu
from functions.geonlm_medians_functions import (
    _robust_center_value,
    _aswmf_spatial_weight,
    _aswmf_local_fallback,
)

%matplotlib inline

## Parametros

Por padrao, usamos o mesmo caso da previa: Set12 `07.png`, ruido sal-e-pimenta `medium`, com o hibrido `f=1`, `t=3`, `nn=7`, `h_multiplier=0.001`.

In [ ]:
pickle_path = PROJECT_ROOT / 'data/output/set12Hibrid/salt_pepper_medium/full_512/results/array_nlm_salt_pepper_medium_filtereds.pkl'
file_name = '07.png'

f = 1
t = 3
nn_count = 7
h_multiplier = 0.001

z_alpha = 1.96
outlier_pixel_alpha = 0.0
reject_impulse_candidates = True
use_aswmf_spatial_weights = True
aswmf_weight_diag_1 = 1.0
aswmf_weight_diag_2 = 1.0
aswmf_weight_other = 10.0

selection_margin = 55
min_center_distance = 80
variance_window = 31

In [ ]:
with pickle_path.open('rb') as fobj:
    records = pickle.load(fobj)

item = next(row for row in records if row['file_name'] == file_name)
noisy = item['img_noisy_salt_pepper_np'].astype(np.float32)
ref = item['img_reference_np'].astype(np.float32)
nlm_h = float(item['nlm_h'])
h = nlm_h * h_multiplier

print(f'Arquivo: {file_name}')
print(f'Pickle: {pickle_path}')
print(f'Densidade sal-e-pimenta: {item["salt_pepper_density"]}')
print(f'nlm_h={nlm_h:.0f}; h={h:.3f}; f={f}; t={t}; nn={nn_count}')
print(f'Imagem ruidosa: shape={noisy.shape}, min={noisy.min():.1f}, max={noisy.max():.1f}')

## Selecao das regioes

Os centros sao escolhidos apenas entre pixels impulsivos (`0` ou `255`) da imagem ruidosa. A classificacao lisa/mista/densa usa a variancia local da imagem de referencia, para evitar que o ruido sal-e-pimenta domine a escolha.

In [ ]:
def select_impulse_regions(noisy, ref, margin=55, min_distance=80, variance_window=31):
    var_map = generic_filter(ref, np.var, size=variance_window, mode='reflect')
    impulse_mask = (noisy == 0.0) | (noisy == 255.0)

    valid_mask = impulse_mask.copy()
    valid_mask[:margin, :] = False
    valid_mask[-margin:, :] = False
    valid_mask[:, :margin] = False
    valid_mask[:, -margin:] = False

    ys, xs = np.where(valid_mask)
    vals = var_map[ys, xs]
    targets = [('lisa', 0.04), ('mista', 0.50), ('densa', 0.96)]
    selected = []

    for label, quantile in targets:
        target = np.quantile(vals, quantile)
        order = np.argsort(np.abs(vals - target))
        chosen = None

        for idx in order:
            y, x = int(ys[idx]), int(xs[idx])
            far_enough = all(
                (x - sx) ** 2 + (y - sy) ** 2 >= min_distance ** 2
                for _, sx, sy, _ in selected
            )
            if far_enough:
                chosen = (label, x, y, float(var_map[y, x]))
                break

        if chosen is None:
            idx = int(order[0])
            chosen = (label, int(xs[idx]), int(ys[idx]), float(var_map[ys[idx], xs[idx]]))

        selected.append(chosen)

    return selected, var_map


selected, var_map = select_impulse_regions(
    noisy,
    ref,
    margin=selection_margin,
    min_distance=min_center_distance,
    variance_window=variance_window,
)

for label, x, y, var in selected:
    print(f'{label:5s}: centro=({x:3d},{y:3d}), var={var:8.2f}, ruidoso={noisy[y, x]:6.1f}, ref={ref[y, x]:6.1f}')

In [ ]:
colors = {'lisa': '#2ca02c', 'mista': '#ff7f0e', 'densa': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(noisy, cmap='gray', vmin=0, vmax=255)

for label, x, y, var in selected:
    color = colors[label]
    ax.add_patch(Rectangle((x - t - 0.5, y - t - 0.5), 2 * t + 1, 2 * t + 1, fill=False, lw=2.2, ec=color))
    ax.scatter([x], [y], s=55, c=color, edgecolors='white', linewidths=0.8)
    ax.text(x + 7, y - 7, label, color='white', fontsize=10, weight='bold', bbox=dict(facecolor=color, edgecolor='none', alpha=0.9, pad=2))

ax.set_title('Regioes selecionadas sobre o ruido sal-e-pimenta medium')
ax.axis('off');

## Grafo local e paths

A funcao abaixo replica o calculo local do GEONLMedians hibrido: extrai os patches na janela `t`, monta o grafo k-NN, roda Dijkstra a partir do patch central, aplica o kernel com `h`, rejeita candidatos impulsivos e usa fallback ASWMF quando `Z=0`.

In [ ]:
img_n = mirror_cpu(noisy.astype(np.float32), f)
m, n = noisy.shape


def graph_for_pixel(row, col):
    im = row + f
    jn = col + f
    rmin = max(im - t, f)
    rmax = min(im + t, m + f)
    smin = max(jn - t, f)
    smax = min(jn + t, n + f)

    n_patches = (rmax - rmin) * (smax - smin)
    patch_size = (2 * f + 1) ** 2
    n_neighbors = min(nn_count, max(1, n_patches - 1))

    dataset = np.empty((n_patches, patch_size), dtype=np.float32)
    pixels_search = np.empty(n_patches, dtype=np.float32)
    valid_candidates = np.ones(n_patches, dtype=bool)
    spatial_weights = np.ones(n_patches, dtype=np.float32)

    source = -1
    k = 0
    for r in range(rmin, rmax):
        for s in range(smin, smax):
            patch = img_n[r - f:r + f + 1, s - f:s + f + 1]
            candidate_center = img_n[r, s]
            dataset[k] = patch.ravel()
            pixels_search[k] = _robust_center_value(patch, candidate_center, z_alpha, outlier_pixel_alpha)

            if reject_impulse_candidates and (candidate_center == 0.0 or candidate_center == 255.0):
                valid_candidates[k] = False

            if use_aswmf_spatial_weights:
                spatial_weights[k] = _aswmf_spatial_weight(
                    r - im,
                    s - jn,
                    weight_diag_1=aswmf_weight_diag_1,
                    weight_diag_2=aswmf_weight_diag_2,
                    weight_other=aswmf_weight_other,
                )

            if r == im and s == jn:
                source = k
            k += 1

    if source < 0:
        source = 0

    graph = nx.from_scipy_sparse_array(
        sknn.kneighbors_graph(dataset, n_neighbors=n_neighbors, mode='distance')
    )
    lengths, paths = nx.single_source_dijkstra(graph, source)

    points = list(lengths.keys())
    distances = np.array(list(lengths.values()), dtype=np.float32)
    similarity_weights = np.exp(-(distances ** 2) / (h ** 2))
    similarity_weights *= valid_candidates[points].astype(np.float32)
    if use_aswmf_spatial_weights:
        similarity_weights *= spatial_weights[points]

    z_value = float(similarity_weights.sum())
    if z_value <= 0:
        filtered = _aswmf_local_fallback(
            img_n,
            im,
            jn,
            radius=f,
            weight_diag_1=aswmf_weight_diag_1,
            weight_diag_2=aswmf_weight_diag_2,
            weight_other=aswmf_weight_other,
        )
    else:
        filtered = float(np.sum(similarity_weights * pixels_search[points]) / z_value)

    tree_edges = set()
    active_edges = set()
    for point, weight in zip(points, similarity_weights):
        path = paths[point]
        for a, b in zip(path[:-1], path[1:]):
            edge = tuple(sorted((a, b)))
            tree_edges.add(edge)
            if weight > 0:
                active_edges.add(edge)

    pos = PCA(n_components=2).fit_transform(dataset)
    if np.allclose(pos.std(axis=0), 0):
        pos = np.column_stack(np.unravel_index(np.arange(n_patches), (rmax - rmin, smax - smin)))[..., ::-1]

    return {
        'graph': graph,
        'source': source,
        'valid': valid_candidates,
        'tree_edges': tree_edges,
        'active_edges': active_edges,
        'weights': dict(zip(points, similarity_weights)),
        'filtered': float(filtered),
        'center_noisy': float(noisy[row, col]),
        'center_ref': float(ref[row, col]),
        'z': z_value,
        'pos': {idx: pos[idx] for idx in range(len(pos))},
    }

## Visualizacao composta

In [ ]:
fig = plt.figure(figsize=(16, 10), constrained_layout=True)
gs = fig.add_gridspec(3, 4, width_ratios=[1.15, 1, 1, 1])

ax_img = fig.add_subplot(gs[:, 0])
ax_img.imshow(noisy, cmap='gray', vmin=0, vmax=255)

for label, x, y, var in selected:
    color = colors[label]
    ax_img.add_patch(Rectangle((x - t - 0.5, y - t - 0.5), 2 * t + 1, 2 * t + 1, fill=False, lw=2.2, ec=color))
    ax_img.scatter([x], [y], s=45, c=color, edgecolors='white', linewidths=0.8)
    ax_img.text(x + 7, y - 7, label, color='white', fontsize=10, weight='bold', bbox=dict(facecolor=color, edgecolor='none', alpha=0.9, pad=2))

ax_img.set_title('07.png com ruido sal-e-pimenta medium\ncentros impulsivos selecionados')
ax_img.axis('off')

for col, (label, x, y, var) in enumerate(selected, start=1):
    info = graph_for_pixel(y, x)
    graph = info['graph']
    pos = info['pos']
    color = colors[label]

    ax_patch = fig.add_subplot(gs[0, col])
    crop = noisy[y - 18:y + 19, x - 18:x + 19]
    ax_patch.imshow(crop, cmap='gray', vmin=0, vmax=255)
    ax_patch.add_patch(Rectangle((18 - t - 0.5, 18 - t - 0.5), 2 * t + 1, 2 * t + 1, fill=False, lw=2, ec=color))
    ax_patch.scatter([18], [18], s=36, c=color, edgecolors='white', linewidths=0.8)
    ax_patch.set_title(f'{label}\ncentro=({x},{y}) var={var:.1f}')
    ax_patch.axis('off')

    ax_graph = fig.add_subplot(gs[1:, col])
    nx.draw_networkx_edges(graph, pos, ax=ax_graph, edge_color='#c9c9c9', width=0.6, alpha=0.45)
    if info['tree_edges']:
        nx.draw_networkx_edges(graph, pos, ax=ax_graph, edgelist=list(info['tree_edges']), edge_color='#6baed6', width=1.15, alpha=0.7)
    if info['active_edges']:
        nx.draw_networkx_edges(graph, pos, ax=ax_graph, edgelist=list(info['active_edges']), edge_color=color, width=2.5, alpha=0.95)

    node_colors = []
    node_sizes = []
    for node in graph.nodes:
        if node == info['source']:
            node_colors.append('#e41a1c')
            node_sizes.append(95)
        elif not info['valid'][node]:
            node_colors.append('#111111')
            node_sizes.append(42)
        else:
            weight = info['weights'].get(node, 0.0)
            node_colors.append(color if weight > 0 else '#f2f2f2')
            node_sizes.append(58 if weight > 0 else 32)

    nx.draw_networkx_nodes(graph, pos, ax=ax_graph, node_color=node_colors, node_size=node_sizes, edgecolors='#333333', linewidths=0.35)
    ax_graph.set_title(
        f'k-NN paths ({label})\n'
        f'nos={graph.number_of_nodes()} arestas={graph.number_of_edges()} Z={info["z"]:.2e}\n'
        f'ruidoso={info["center_noisy"]:.0f} ref={info["center_ref"]:.1f} filtrado={info["filtered"]:.1f}'
    )
    ax_graph.axis('off')

fig.suptitle(
    f'Set12 07.png | sal-e-pimenta medium | f={f}, t={t}, nn={nn_count}, h={h:.3f} '
    f'(nlm_h={nlm_h:.0f} x {h_multiplier})',
    fontsize=14,
);

## Opcional: salvar a figura

Execute a celula abaixo somente quando quiser persistir a visualizacao.

In [ ]:
# output_path = PROJECT_ROOT / 'data/output/set12Hibrid/salt_pepper_medium/full_512/results/set12_07_medium_paths_graphs.png'
# fig.savefig(output_path, dpi=300, bbox_inches='tight')
# output_path